# Model Evaluations
*Why AI engineers need model evals, the four-step evaluation process, benchmarks vs. custom evals, the eight core model capabilities in depth, the anatomy of a benchmark, eval harnesses, and four reasons to distrust benchmark numbers*

## Why AI engineers need model evals

Model evals exist, in general, to measure the capabilities of an LLM — and a well-known principle applies directly here: capabilities that can't be measured can't be reliably improved. That's a fairly obvious reason for a frontier lab to care about model evals, since a lab's own training decisions depend on knowing exactly where a model is falling short. The more interesting question is different: someone building LLM-based applications, not training models, never runs the kind of large-scale benchmarking a frontier lab does — so why should model evals matter in that day-to-day work at all?

There are four concrete reasons.

### Reason 1: choosing between candidate models

Building any LLM-based application starts with a basic decision: which underlying model should power it? "Whichever one, they're both fine" isn't an answer a professional setting accepts — a real decision needs concrete justification for picking one candidate over another. Model evals are exactly what supplies that justification: they identify which capabilities actually matter for the application at hand, and which candidate model scores better on those specific capabilities. Without them, a choice between two models is just a guess dressed up as a decision.

### Reason 2: verifying that a new model release is an actual improvement

A production application is often already running on some specific model. When that provider ships a newer model, deciding whether to switch shouldn't be based on marketing claims or a vague sense that "newer is better." Model evals are what let that claim actually be checked — running both the current and the candidate model through the same evaluation and comparing the results directly, rather than assuming an upgrade is real without verifying it.

### Reason 3: assessing whether a model is safe to use

Before deploying any model, it's worth knowing how it behaves under pressure — how often it hallucinates, and how resistant it is to jailbreak attempts trying to make it behave outside its intended boundaries. This is exactly the kind of thing model evals are built to measure, independent of any specific application built on top of the model.

### Reason 4: deciding whether to self-host or use a proprietary API

A meaningful architectural decision sits behind almost every LLM-based application: use a proprietary model through its provider's API, or pull an open-source model and host it independently. Both paths have real trade-offs — a proprietary model is often more capable but costs more per call and creates a dependency on an external provider; a self-hosted open-source model can be cheaper at scale and gives full control, but may not match the proprietary option's capabilities. Making that trade-off concrete, rather than guessing at it, is again something model evals directly support.

Put together, these four reasons are why model evals matter for someone building applications, not just for the labs training the underlying models — without them, there's no way to justify a model choice, verify that an upgrade is real, assess safety, or make an informed build-vs-buy call.

## What a model eval formally is

A model eval is a systematic process for measuring an underlying model's capabilities, behavior, reliability, and operational characteristics under controlled conditions. The core idea is unchanged from the simpler version covered earlier — a model eval is a way to test an LLM — but this fuller definition makes explicit that it covers more than raw capability alone: how the model behaves, how reliably it performs, and its operational characteristics like cost and latency all fall under the same umbrella.

## The four steps every model eval follows

Regardless of what specifically is being tested, every model eval follows the same four steps:

1. **Decide which capability to test.** LLMs are general-purpose — there's no single model eval that tests everything at once, unlike a human IQ score that tries to summarize general ability in one number. Reasoning, coding, safety, and instruction-following each need their own separate eval.
2. **Bring a test.** A specific mechanism or test is chosen to actually measure the capability decided on in step one.
3. **Run the model under a fixed protocol.** The model is run against that test under controlled, fixed conditions — the same prompting approach, the same settings — so the process is repeatable and, when comparing multiple models, every model gets evaluated under identical conditions.
4. **Score and interpret.** Once the test is complete, the result gets scored and interpreted.

These four steps are true of any model eval, whether it's a big public benchmark or something built for one specific use case.

## Two types of tests: benchmarks and custom evals

Step two above — bringing a test — can be satisfied in two different ways:

- **Benchmarks.** A standardized, shared test that everyone runs the same way — MMLU and SWE-bench are examples covered earlier. Because everyone uses the same test, benchmarks are excellent for comparing models on common ground, and their scores are widely recognized and cited.
- **Custom evals.** A test assembled from an application's own real tasks and data, measuring specifically what that application cares about rather than what's generically useful across every possible use case.

It's a reasonable question why a custom eval would ever be needed if benchmarks already report how good a model is at math, reasoning, coding, and so on. The answer becomes clear with a concrete comparison.

## A worked example: choosing a model for email classification

Recall the email-classification system from earlier — a system that reads incoming support emails and routes them to the right team (billing, technical, or refund-related, say). Suppose there are two candidate models to build it on:

- **Model A** — a large, powerful model near the top of most public benchmark leaderboards, but expensive: roughly $15 per million tokens.
- **Model B** — a smaller, cheaper model, sitting in the middle of the same benchmark leaderboards rather than the top: roughly $0.50 per million tokens.

On pure benchmark performance, Model A wins across the board — better at math, coding, language generation, everything a general benchmark measures. Common sense might say the decision is already made: just use Model A. But committing to that without checking further can be expensive at real scale.

## Running a custom eval on the actual task

Rather than relying on benchmark scores alone, a golden dataset gets built directly from the task at hand: 200 to 500 real past emails, each manually labeled with its correct category (and, say, its urgency level). Both models get run against this same dataset, and the results look like this:

| Metric | Model A | Model B |
|---|---|---|
| Classification accuracy | 94% | 91% |
| Urgency accuracy | 88% | 87% |
| Cost per 1,000 emails | ~$6.00 | ~$0.21 |
| Latency per request | ~4.1 seconds | ~0.9 seconds |

Model A is still slightly more accurate on both measures — but only slightly. On the task that actually matters here, Model B is nearly as good, while costing roughly 30 times less and responding more than four times faster.

## Why the custom eval changes the decision

Based on this table, Model B is the stronger choice for this specific application: the small accuracy gap barely matters for classifying support emails, while the cost and latency gap is substantial at any real volume. Relying on public benchmarks alone would never have surfaced this — Model A outscores Model B on every benchmark, and a benchmark-only comparison would point unambiguously toward Model A every time. Only by building a custom eval around the actual task and the actual data did it become clear that Model B was the better value for this particular use case.

This is the core reason custom evals matter alongside benchmarks: benchmarks reveal a model's generic capabilities, but only a custom eval, built from an application's own data, reveals which model actually performs best for that specific application.

## The eight core capabilities, in depth

Before going further into benchmarks specifically, it's worth understanding *what* they're actually measuring. LLMs are general-purpose — the same model can generate text, analyze sentiment, summarize a document, or tag parts of speech — which means no single test can capture everything they can do. In practice, the field has converged on eight core capability categories, and nearly every benchmark in wide use targets one of these eight.

### 1. Knowledge and reasoning

This capability bundles two closely related things: how much factual knowledge a model absorbed during training, and whether it can connect that knowledge together into a valid conclusion.

**What gets measured:**
- **Factual recall** across subjects — biology, physics, chemistry, history, and so on. MMLU, covered earlier, is the benchmark most associated with this: 57 subjects, testing whether a model gets the basic, important questions in each one right.
- **Multi-step logical reasoning** — whether a model can chain multiple facts together, in the right order, to reach a conclusion. A task like "summarize the full arc of human evolution from the earliest point through today, and explain why society looks the way it does now" tests both at once: the model needs accurate factual recall of what happened and in what order, and it needs to reason about how those events connect to produce today's outcome.

**Why it matters to frontier labs:** performance here is treated as a rough proxy for how intelligent a model is overall, which is why it's one of the most closely watched and competed-over categories.

**Real-world relevance:** a research assistant that analyzes papers and helps develop new ideas draws directly on both knowledge and reasoning. So does any application working with technically dense documents — asking questions against an uploaded machine learning paper, for instance — or any chatbot built to assist a professional in their field, like a lawyer or a teacher.

### 2. Coding and software engineering

From a purely economic standpoint, this may be the single most consequential capability category — AI coding tools have become one of the fastest-growing and most highly valued categories in software, precisely because models that can code well create enormous value for engineering teams.

**What gets measured:**
- **Functional code generation** — given a plain-English description, can the model produce a function that actually works?
- **Test generation and self-correction** — can it write test cases for its own code, and use failing tests to improve that code?
- **Bug fixing** in an existing codebase, not just fresh code.
- **Multi-file, long-horizon engineering tasks** — handed an entire codebase and asked to refactor it along some specific dimension, can the model actually carry that out?
- **Command-line proficiency** — running multiple sequential commands correctly, such as installing packages, configuring a server, and setting up an environment.
- **API and function calling.**

**Real-world relevance:** this is exactly the capability set behind the current wave of AI coding agents — any application in that category depends directly on how strong a model is across these specific sub-skills.

### 3. Mathematics

Mathematics is really a specific, more rigorous form of reasoning — reaching a solution through a precise, verifiable, step-by-step process — which is why it gets tracked as its own capability rather than folded entirely into general reasoning.

**What gets measured**, roughly in increasing order of difficulty:
- **Grade-school mathematics** — problems at a basic school level.
- **Competition-level problem solving** — olympiad-style problems that require creative, non-obvious approaches.
- **Undergraduate-level problems.**
- **Research-level mathematical reasoning** — open-ended problems without a known solution yet, testing whether a model can make genuine progress rather than just retrieving a memorized method.

**Real-world relevance:** this capability underlies applications in scientific computing, financial modeling, engineering simulation, and data analysis — any domain where the answer needs to be not just plausible-sounding but numerically or symbolically correct.

### 4. Long context

This domain measures whether a model can actually make effective use of information from very long inputs — sometimes hundreds of thousands of tokens — rather than just technically accepting them.

**What gets measured:**
- **Fact extraction** — pulling out one small, specific fact buried somewhere in a very long input.
- **Entity retrieval** — fetching details about one particular person or entity out of a large document.
- **Long-context summarization.**
- **Codebase context maintenance** — for a coding agent specifically, whether it can keep track of a large codebase's context without losing track of earlier parts.

**Why it matters:** advertised context window sizes — 128K, 200K, a million tokens — describe how much a model can technically accept as input, not how well it actually uses all of it. In practice, as a conversation or document grows longer, a model's ability to make full use of everything in its context tends to degrade, and response quality drifts down along with it. This capability specifically tests that gap between advertised and effective context.

**Real-world relevance:** this applies to essentially any LLM-based application — long-context capability is what determines how much can genuinely be handed to a model at once before quality starts to suffer.

### 5. Vision and multimodal

This domain moves past text entirely: does the model understand images and video, not just written language?

**What gets measured:** whether a model can interpret visual input correctly — recognizing what's in an image, understanding a video, and reasoning about visual content the same way it would reason about text.

**Real-world relevance:** everyday use increasingly spans more than text alone — pointing a camera at the contents of a fridge and asking what could be cooked with them, or scanning shelves in a library to find a specific book. Multimodal capability is what makes interactions like these possible at all.

### 6. Agentic and tool use

This capability is becoming increasingly central, and it's essentially where the entire field of agentic AI emerged from: a model that can only print text back at a user is fundamentally limited compared to one that can actually take action by using tools.

**What gets measured:**
- Whether the model can browse the web on its own.
- Whether it can perform **structured tool calling** reliably.
- Whether it can interact correctly with external **APIs**.
- Whether it can operate a **desktop or computer environment** directly.

**Real-world relevance:** any agentic application — one that needs to actually complete tasks rather than just describe how to do them — depends directly on how reliably the underlying model handles this category.

### 7. Safety and alignment

This domain measures whether a model can be trusted to behave responsibly, independent of how capable it otherwise is.

**What gets measured:**
- Whether the model avoids generating **harmful content**.
- **Resistance to adversarial attacks** — how easily it can be manipulated into behaving outside its intended boundaries.
- **Truthfulness versus sycophancy** — whether a model gives an honest, critical assessment when asked to evaluate an idea, rather than reflexively agreeing with and praising whatever the user proposes. A model that says "here's a flaw in that reasoning" is behaving more truthfully than one that responds with uncritical enthusiasm to everything it's shown.
- More recently, **cybersecurity-related skills** specifically — cryptography, reverse engineering, digital forensics — have become their own benchmark category, since a handful of recent frontier models have scored unusually high in this area, including surfacing real vulnerabilities in existing software during testing.

**Why it matters to frontier labs:** safety carries both regulatory pressure and serious reputational risk — a single notable safety incident can be damaging enough to threaten an AI lab's entire business, which is part of why this category gets so much dedicated attention.

### 8. Instruction following

This category is somewhat underrated relative to how much it actually matters in practice: does a model do exactly what it was asked to do, in exactly the way it was asked to do it?

**What gets measured:**
- Whether format instructions are followed — answering in a bullet list when asked for one, for instance.
- Whether length constraints are respected — staying under a specified word count.
- Whether tone instructions are honored — a friendly tone when asked for one, and so on.
- Whether the model asks a **clarifying question** when the user's instructions are genuinely ambiguous, rather than guessing.

**Real-world relevance:** this translates directly into user satisfaction. A model that doesn't reliably follow instructions produces a frustrating product experience regardless of how capable it is elsewhere — which is exactly why this capability gets measured as its own category rather than assumed to come for free with general intelligence.

## The eight, at a glance

Knowledge and reasoning, coding and software engineering, mathematics, long context, vision and multimodal, agentic and tool use, safety and alignment, and instruction following — these eight categories are what nearly every benchmark in circulation targets. Any new benchmark encountered going forward is very likely testing one specific slice of one of these eight, which makes this list a useful map for placing new terminology as it comes up.

## What a benchmark actually is

A benchmark is a standardized test used to measure one specific model capability. That's the whole definition — but a specific structure sits underneath every benchmark, made up of four components: a **dataset and task**, a **run configuration**, a **scoring method**, and an **aggregation method**. Walking through a real benchmark makes each of these concrete.

## A running example: GSM8K

GSM8K is a good benchmark to unpack this with — it's an older, foundational one (2021) covered earlier as the standard math capability benchmark, and its structure is simple enough to see all four components clearly. The name breaks down directly: **G**rade **S**chool **M**ath, with a dataset of roughly **8K** rows.

## Component 1: dataset and task

A benchmark's dataset is a question-and-answer pair set — effectively a golden dataset — plus a defined task describing what the model is supposed to do with each question.

GSM8K's dataset consists of roughly 8,500 grade-school-level math word problems, each paired with its correct numeric answer. A representative example:

> Natalia sold clips to 48 friends in April, and then sold half as many clips in May. How many clips did she sell altogether?

The answer: 48 in April, half that (24) in May, for a total of 72. The task tied to this dataset is simple to state: given a grade-school-level math word problem, generate the correct answer. Every benchmark defines its dataset and task this way — a set of questions with known-correct answers, and a clear statement of what the model being tested is asked to do with each one.

## Component 2: run configuration

A benchmark only produces comparable numbers if every model is tested under identical conditions — if two models are run with different settings, their scores aren't actually comparable to each other. The run configuration is what fixes those conditions in advance, and it has three parts.

**Prompt construction.** This decides whether the model sees the question alone (**zero-shot**) or a handful of worked examples solved in the prompt before the actual question (**few-shot**) — few-shot generally improves performance by showing the model the expected pattern first. It also decides whether **chain-of-thought** reasoning is invited — a system prompt instructing the model to work through the problem step by step, rather than jumping straight to an answer — which tends to reduce errors on multi-step problems. GSM8K is classically reported using an 8-shot, chain-of-thought setup: eight worked examples shown first, with the model encouraged to reason step by step before answering.

**Decoding and sampling configuration.** Temperature is typically set close to zero for a benchmark like this, since a temperature above zero introduces creative variation that makes results inconsistent across runs. A maximum token limit is also fixed in advance — too low, and a chain-of-thought response can get cut off mid-reasoning before it reaches an answer; too high, and a stronger model gets more room to reason than a fixed comparison should allow. The same limit applies to every model being compared.

**Tool-use policy.** Whether external tools are permitted during evaluation is also decided upfront. GSM8K doesn't allow tools — a model can't offload the arithmetic to a calculator or code interpreter, since the point is testing the model's own reasoning. Other benchmarks make the opposite choice deliberately: SWE-bench, which tests whether a model can resolve real GitHub issues, requires tool access, since fetching an actual issue from a repository isn't possible without one.

## Component 3: scoring method

Scoring happens in two stages: extracting the actual answer from the model's raw output, and then comparing that extracted answer against the known-correct one.

**Extraction.** An LLM's raw output isn't guaranteed to come back in a clean, predictable format — a model might respond "The answer is 72," or just "72," or embed the number somewhere inside a longer explanation. Before anything can be scored, the actual answer has to be pulled out of that free-form text reliably, typically using structured output enforcement or pattern matching, so that what gets compared is just the number itself — 72 — not the surrounding text.

**Comparison.** For a benchmark like GSM8K with a single, unambiguous correct answer, comparison is a straightforward exact match: the extracted answer either equals 72 or it doesn't, scored as 1 or 0. Benchmarks with **closed-ended** answers — a single number, a specific label — can be scored this mechanically. Benchmarks with **open-ended** answers — a full paragraph rather than a single value — can't be checked this way, since two differently worded paragraphs can both be correct. Those require an LLM-as-judge (or, less commonly, a human reviewer) to assess whether the answer is right, rather than a programmatic exact match.

**Scoring strategy also varies in how many attempts count.** A few common conventions:
- **Pass@1** — the model gets one attempt at a question, and it's simply right or wrong.
- **Pass@k** — the model is asked the same question *k* times, and it counts as correct if at least one of those *k* attempts is right. This is a more lenient standard than pass@1.
- **Majority@k** — the model is asked the same question *k* times, and whichever answer appears most often across those *k* attempts is treated as the model's answer.

A reported benchmark score of, say, 82% means very different things depending on which of these was used — which is exactly why it's worth checking which scoring strategy sits behind any number before comparing it to another model's score on the same benchmark.

## Component 4: aggregation method

Once every individual question has been scored as correct or incorrect, those individual results need to be rolled up into one final number.

For a benchmark like GSM8K, this is direct: out of, say, 1,000 questions, if the model answered 920 correctly, the aggregate score is simply 920/1000 = 92%.

Aggregation isn't always this simple, though. MMLU, which tests general knowledge across 57 subjects, reports results per subject first — 87% on biology, 91% on physics, 72% on economics, and so on — and then needs to combine those 57 numbers into one overall score. Averaging the raw percentages equally isn't necessarily correct here, since the underlying dataset may not have the same number of questions in every subject; a subject with far more questions arguably deserves more weight in the final number, which can call for a weighted mean rather than a simple average. Different benchmarks use different aggregation strategies depending on exactly this kind of structural detail in their dataset.

## Where all of this is documented

Nearly every widely used benchmark started life as a research paper, and that paper is where its dataset, run configuration, scoring method, and aggregation method are all formally specified. The GSM8K paper, for instance, lays out exactly what dataset it uses, what prompting setup, what scoring convention, and how results get aggregated — and the same structure holds for MMLU, SWE-bench, and effectively every other benchmark in circulation. Researchers in a given area design a benchmark specifically to measure a capability they care about, publish it as a paper alongside its dataset, and if it's well designed, that benchmark and dataset become the standard the rest of the field measures new models against.

## Running a benchmark, step by step

Putting all four components together, evaluating a model against a benchmark is fundamentally a loop — one iteration per row in the benchmark's dataset. Concretely, imagine a newly trained model being tested against GSM8K for the first time. For every question in the dataset, the loop does the same thing:

1. **Load the item** — pull the next question from the dataset.
2. **Build the prompt** — insert the question into the exact prompt template the benchmark specifies, including any few-shot examples (eight worked examples, for GSM8K) and the chat template being used.
3. **Call the model** with the fixed decoding configuration — the specified temperature, max tokens, and any stopping criteria — all set exactly as the run configuration requires.
4. **Capture the raw output** and **extract the answer** from it — pulling out just the number, say, from whatever free-form text the model returned.
5. **Score** that extracted answer against the known-correct one, storing the result (1 or 0) in a running list.

This repeats once for every row in the dataset — 8,500 times, for GSM8K's full set. Once the loop finishes, the stored list of individual scores gets aggregated into the model's final benchmark score.

## Why this "simple loop" isn't actually simple

Described this way, the process sounds straightforward enough to implement in an afternoon — load a question, call the model, check the answer, repeat. Actually writing the code reveals a different story. A handful of things that look trivial in pseudocode turn out to require real engineering:

- **Robust answer extraction.** A model might return "The answer is 72," or just "72," or bury the number inside a longer explanation — reliably pulling out the right value across thousands of differently formatted responses needs real extraction logic, not a simple string match.
- **Following the exact scoring mechanism** specified in the benchmark's original paper, which can have its own quirks and edge cases.
- **Batching**, since sending thousands of individual model calls one at a time is far too slow to be practical.
- **Retry logic** for when an individual API call fails partway through a run of thousands of calls.
- **Rate limit handling**, so a large evaluation run doesn't simply fail partway through.

The loop itself is conceptually simple. Running that loop *reliably*, at the scale of thousands of model calls, needs considerably more engineering built up around it.

## Eval harnesses: not writing all of that from scratch

Because of that gap between the simple idea and the engineering reality, this "plumbing" code is rarely written from scratch. An **eval harness** is a piece of software written specifically to execute model evaluations — handling extraction, batching, retries, rate limits, and everything else described above — so that running a benchmark doesn't require reimplementing all of it every time.

A useful analogy: a benchmark is like an exam paper, and an eval harness is like the administration that actually runs the exam — checking people in, timing it, collecting the answer sheets — handling everything around the paper itself so nobody sitting the exam has to manage that logistics personally.

A few widely used eval harnesses:

- **lm-evaluation-harness**, from EleutherAI — a widely adopted framework supporting dozens of standard academic benchmarks, and the backend behind several public leaderboards.
- **Inspect**, from the UK AI Security Institute — built for both capability and safety evaluations, including agentic tasks that involve tool use.
- **HELM** (Holistic Evaluation of Language Models), from Stanford's Center for Research on Foundation Models — a framework aimed at broad, standardized, reproducible evaluation across many benchmarks and models at once.

Without tools like these, every team evaluating a model would need to write its own version of this plumbing, and there'd be no guarantee that two different implementations of "the same" benchmark actually produce comparable numbers. Eval harnesses are what keep results consistent and reproducible across the field, rather than depending on how carefully any one team happened to implement the details.

## A concrete example: running GSM8K through an eval harness

Running an actual benchmark through a harness like lm-evaluation-harness illustrates just how much complexity gets abstracted away. After installing the library and providing an API key for the provider being tested, evaluating a candidate model against GSM8K comes down to a single command specifying:

- **Which model** to evaluate.
- **How many concurrent requests** to allow at once.
- **Which task** (benchmark) to run — GSM8K, in this case.
- **Chain-of-thought** enabled, with **8-shot** prompting, matching GSM8K's standard run configuration.
- A **limit** on how many questions to actually run — useful for a quick sanity check before committing to the full dataset. Running all 8,500 questions costs meaningfully more than running a small sample; a full GSM8K run might cost roughly ₹2,300, against roughly ₹3–4 for a 20-question sample used just to confirm the setup works.
- An **output path** for the results.

No loop, no manual API calls, no extraction logic written by hand — the single command handles all of it. Running it against a 20-question sample produced 18 correct answers, a 90% score, with every individual question, its model-generated answer, and its correctness recorded as a structured result file. Removing the question limit and running the full dataset would produce a genuine, publishable GSM8K score for that model — the same basic process a frontier lab uses internally to report its own benchmark numbers.

## Choosing an eval harness: benchmarking vs. application evals

Not every eval harness is built for the same job. DeepEval, for instance, also supports running standard benchmarks like GSM8K, but requires noticeably more code to actually run a model against one — it's built with a stronger focus on evaluating custom LLM-based applications rather than running large-scale public benchmarks efficiently, and its benchmark support tends to be limited to a set of well-established ("saturated") options. lm-evaluation-harness, by contrast, is the more standard choice specifically for running public benchmarks — a single command rather than substantial custom code — which is why it tends to be the default choice, including at scale inside larger companies.

## Who actually runs model evaluations

Since most benchmarks are public — their papers, methodology, and datasets are freely available online — anyone technically can run one. In practice, three categories of people actually do:

**Frontier labs themselves.** This is the most obvious case. Labs run benchmarks throughout model development, not just at the end — checkpoints get pulled out and benchmarked at different points during training, giving a running signal on whether training is heading in the right direction, and letting course corrections happen mid-training rather than only after it's finished. Benchmarks also inform **release gating** — deciding whether a new model version is actually better than the last one and therefore worth shipping — and they double as a **marketing channel**: a model that posts an unusually strong score on a specific benchmark tends to get organically covered and discussed well beyond the lab's own announcement.

**Third-party evaluators.** Independent organizations and leaderboards — LM Arena is a well-known example — exist specifically to benchmark different models under identical, neutral conditions and publish the resulting rankings. Because their entire business is running evaluations fairly rather than promoting any one model, their numbers tend to be more trustworthy than a lab's own self-reported scores, and they often report information labs don't — like cost and latency alongside raw capability scores.

**AI engineering teams themselves.** Rather than relying entirely on either of the above, teams building applications often run their own evaluations using the same public benchmarks and open tools like eval harnesses, testing candidate models under their own specific conditions to see exactly how they perform, what they cost, and how fast they respond.

## Why lab-published benchmark numbers deserve skepticism

When choosing a model for an application, self-reported benchmark numbers from the lab that built the model shouldn't be taken entirely at face value. A fair question to ask is simply who ran the evaluation — if the answer is "the lab itself, under its own controlled settings," some skepticism is warranted.

The same logic applies here as it does to a car's advertised fuel efficiency: manufacturer figures are usually measured under ideal, favorable conditions, and real-world mileage tends to fall short of the advertised number. Lab-published benchmark scores work similarly — they represent something closer to a ceiling, achieved under conditions set up to be as favorable as possible, rather than what a typical real-world use case will actually see.

**Cherry-picking** compounds this: labs tend to prominently feature the specific benchmarks their model happens to score well on, while quietly downplaying or omitting the ones where it doesn't. A model can look exceptionally strong from the announcement alone, purely because the announcement was curated to show its best results — which is exactly why relying on independently run, neutral evaluations matters more than trusting a headline number from the model's own creator.

## A limitation of benchmarks: contamination

Benchmarks aren't flawless measurement instruments, and their results shouldn't be trusted blindly. The first and most significant limitation is **benchmark contamination**.

Most well-known benchmarks — MMLU, GSM8K, and others — are public: their research papers, methodology, and datasets are all openly available on the internet, and have been for years in the case of older benchmarks from 2021–2023. Modern frontier models are trained on massive scrapes of internet text, and there's a real chance that a public benchmark's questions *and their correct answers* end up directly inside a model's training data simply because that data was sitting on the public internet when the scrape happened.

If a model has effectively seen a benchmark's exact questions and answers during training, a high score on that benchmark doesn't necessarily reflect genuine reasoning ability — it may just reflect memorization. This is a real, well-recognized problem, especially for older public benchmarks that have had years to be scraped, quoted, and republished across the web. Two common mitigations: using **private benchmarks** whose questions were never made public in the first place, and using **dynamic benchmarks**, whose dataset gets refreshed on a rolling basis rather than staying fixed indefinitely — unlike a static benchmark, a moving target is much harder to have inadvertently memorized during training.

## A second limitation: saturation

**Benchmark saturation** is a natural process most widely used benchmarks eventually go through. When a benchmark is first introduced, it's genuinely difficult — no model has seen anything like it before, and scores tend to cluster low, in the 25–35% range across most models. Over time, as newer model generations improve, average scores on that same fixed benchmark climb — 35%, then 50%, then 70% — until eventually most current models land in the 90–95%+ range and cluster tightly together, with little separating a top model from a merely good one.

At that point, the benchmark has stopped being useful for its original purpose: if every model scores roughly the same, it can no longer meaningfully distinguish a stronger model from a weaker one. GSM8K, MMLU, and SWE-bench have all reached this point — modern frontier models routinely score in the mid-90s or higher on each, which is exactly why they're treated as saturated rather than cutting-edge today. The typical lifecycle: a benchmark is introduced and is genuinely hard, models improve against it over successive generations, scores compress toward the ceiling, and the benchmark eventually gets retired and replaced by a new, harder one that can once again separate strong models from average ones.

## A third limitation: configuration gaming

**Configuration gaming** happens when a benchmark's run configuration — the settings meant to keep comparisons fair — gets quietly manipulated to favor one specific model. A frontier lab self-reporting its own benchmark score controls every detail of how that evaluation was actually run, and nothing stops it from tilting those details in its own model's favor: allowing its model a tool a fair comparison wouldn't (like a Python interpreter during a math benchmark that's meant to test the model's own reasoning), using different reasoning-effort or temperature settings than what a neutral run would use, and so on. Changes like these can shift a reported score by 5–10 percentage points or more — enough to meaningfully change how impressive a number looks, without changing anything about the model's actual underlying capability.

This is exactly why a headline number like "our model scored 92% on this benchmark," announced without any detail on the exact configuration used — what tools were allowed, what temperature, what reasoning settings, what it cost, how long it took — deserves real skepticism. Without that detail, there's no way to know whether the number reflects genuine capability or a favorably tuned test.

## A fourth limitation: selective aggregation

The final major limitation sits in the aggregation step itself. A benchmark like MMLU reports scores across 57 separate subjects, and a model can score strongly overall while being genuinely weak in one specific subject that happens to matter enormously for a particular application. A model provider reporting only the single aggregate MMLU score has no obligation to also disclose that the model happens to perform poorly in, say, economics specifically — the strong scores across the other 56 subjects can comfortably absorb one weak spot without moving the headline number much at all.

Someone building an economics-focused chatbot, seeing only a strong overall MMLU score, would have no way to know about that specific weakness until deploying and observing poor real-world performance directly. The aggregate number was accurate — it just wasn't the number that mattered for that particular use case, and nothing in the headline score surfaced that gap.

## Why benchmark numbers need a pinch of salt

Put together, contamination, saturation, configuration gaming, and selective aggregation are the four major reasons a reported benchmark number shouldn't be accepted at face value. None of this means benchmarks are useless — they remain a genuinely useful signal of a model's general capability — but a number handed over without visibility into how it was produced isn't something to build a real decision on directly.

This is exactly the gap custom evals are built to close, as covered earlier: rather than trusting a self-reported, possibly favorable, possibly stale, possibly aggregated-into-invisibility benchmark score, building a small evaluation around the actual task at hand — on real data, under conditions that are actually known and controlled — is what makes a model choice a genuinely informed one rather than a guess borrowed from someone else's marketing material.